<h1> Bronze GTFS<h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

<h3>Imports<h3>

In [ ]:
import sys
import gc
from pyspark.sql import DataFrame as SparkDF
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
import requests
from pyspark.sql.types import FloatType
from pyspark.sql import Row
api_key = os.getenv("WARSAW_API_KEY")
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual

<h3>Downloading GTFS<h3>

In [3]:
gtfs_url = "https://gtfs.ztm.waw.pl/last" 
zip_path = "/tmp/warsaw_gtfs.zip"
extract_path = "/tmp/gtfs_data"
urllib.request.urlretrieve(gtfs_url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("GTFS data downloaded and extracted successfully")

GTFS data downloaded and extracted successfully


<h3>Creating Bronze schema<h3>

In [4]:
conn = psycopg2.connect(dbname="Warsaw_Bus_DB", user="admin", password="admin", host="postgres")
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS bronze;")
conn.commit()
cur.execute("CREATE SCHEMA IF NOT EXISTS bronze;")
conn.close()

<h2>Stop_Times Table<h2>

<h3>Reading csv<h3>

In [ ]:
df_stop_times = spark.read.csv(f"{extract_path}/stop_times.txt", header=True)
print(df_stop_times)

['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled', 'timepoint']


<h3>Write to DB<h3>

In [31]:
df_stop_times.write.jdbc(url=jdbc_url, table="bronze.stop_times", \
                         mode="overwrite", properties=db_properties)

<h3>Check<h3>

In [32]:
df_stop_times_test = spark.read.jdbc(url=jdbc_url,
                        table="bronze.stop_times",
                        properties=db_properties)

In [ ]:
assertDataFrameEqual(df_stop_times,df_stop_times_test)

DataFrame lentgths, are the same 6929224 rows


<h2>Stops Table<h2>

<h3>Reading csv<h3>

In [34]:
df_stops = spark.read.csv(f"{extract_path}/stops.txt", header=True)
print(df_stops)

DataFrame[stop_id: string, stop_code: string, stop_name: string, stop_lat: string, stop_lon: string]


<h3>Write to DB<h3>

In [35]:
df_stops.write.jdbc(url=jdbc_url, table="bronze.stops", \
                         mode="overwrite", properties=db_properties)

<h3>Check<h3>

In [36]:
df_stops_test = spark.read.jdbc(url=jdbc_url,
                        table="bronze.stops",
                        properties=db_properties)

In [ ]:
assertDataFrameEqual(df_stops,df_stops_test)

DataFrame lentgths, are the same 6887 rows


<h2>Routes Table<h2>

<h3>Reading csv<h3>

In [38]:
df_routes = spark.read.csv(f"{extract_path}/routes.txt", header=True)
print(df_routes)

DataFrame[route_id: string, agency_id: string, route_short_name: string, route_long_name: string, route_desc: string, route_type: string]


<h3>Write to DB<h3>

In [39]:
df_routes.write.jdbc(url=jdbc_url, table="bronze.routes", \
                         mode="overwrite", properties=db_properties)

<h3>Check<h3>

In [40]:
df_routes_test = spark.read.jdbc(url=jdbc_url,
                        table="bronze.routes",
                        properties=db_properties)

In [ ]:
assertDataFrameEqual(df_routes,df_routes_test)

DataFrame lentgths, are the same 2807 rows


<h2>Calendar Table<h2>

<h3>Reading csv<h3>

In [42]:
df_calendar = spark.read.csv(f"{extract_path}/calendar.txt", header=True)
print(df_calendar)

DataFrame[service_id: string, monday: string, tuesday: string, wednesday: string, thursday: string, friday: string, saturday: string, sunday: string, start_date: string, end_date: string]


<h3>Write to DB<h3>

In [43]:
df_calendar.write.jdbc(url=jdbc_url, table="bronze.calendar", \
                         mode="overwrite", properties=db_properties)

<h3>Check<h3>

In [44]:
df_calendar_test = spark.read.jdbc(url=jdbc_url,
                        table="bronze.calendar",
                        properties=db_properties)

In [ ]:
assertDataFrameEqual(df_calendar,df_calendar_test)

DataFrame lentgths, are the same 9 rows


<h2>Trips Table<h2>

<h3>Reading csv<h3>

In [46]:
df_trips = spark.read.csv(f"{extract_path}/trips.txt", header=True)
print(df_trips)

DataFrame[route_id: string, service_id: string, trip_id: string, trip_headsign: string, direction_id: string, shape_id: string]


<h3>Write to DB<h3>

In [47]:
df_trips.write.jdbc(url=jdbc_url, table="bronze.trips", \
                         mode="overwrite", properties=db_properties)

<h3>Check<h3>

In [48]:
df_trips_test = spark.read.jdbc(url=jdbc_url,
                        table="bronze.trips",
                        properties=db_properties)

In [ ]:
assertDataFrameEqual(df_trips,df_trips_test)

DataFrame lentgths, are the same 242717 rows


<h2>Stop Spark<h2>

In [ ]:
spark.catalog.clearCache()
gc.collect()
spark.stop()